In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os 

load_dotenv()


True

### 면접 질문 생성

In [ ]:
OPEN_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key = OPEN_API_KEY)

# 도메인 예시 

query = '데이터분석가 및 AI 서비스 관련 직무' #교정할 대상 기사제목(원본)

system_instruction = '당신은 아주 깔끔하고 꼼꼼한 데이터분석을 수행하는 회사의 부사장이야, 아주 친절하고 어드바이스를 잘해주는 마음넓은 임원이야' # 지시 어조
user_message = f'{query} 직무에 대한 2차 임원 면접에서 나올 질문을 3개를 생각해서 상투적인 (ex 알겠습니다.)말은 제외하고 면접관의 말투처럼 바로 질문을 ,로 구분해서 출력해줘' # 최종적인 질의
response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = [{
        'role':'system',
        'content' : [{
            'type':'text',
            'text':system_instruction
        }]},
    {
        'role':'user',
        'content' : [{
            'type':'text',
            'text':user_message
        }]}
    ],
    
    response_format = {'type':'text'},
    temperature = 1.0, 
    top_p = 1,
    max_tokens = 2048, # 응답용 최대 토큰 수
    frequency_penalty = 0, # 출력 토큰 빈도 제약
    presence_penalty = 0 # 출력 토큰 재사용 제약
)

# 질문 리스트에 저장
print(response.choices[0].message.content.split(','))
input_text = response.choices[0].message.content.split(',')

['본인의 금융상품 이해도를 평가할 수 있는 질문', ' 위기 상황에서 고객 응대를 어떻게 처리할 것인지 구체적인 사례를 들어 설명해달라는 질문', ' 팀 내 갈등 상황을 경험했을 때 어떻게 조정하고 협력했는지 알려달라는 질문']


###### 면접 질문 생성 결과(OpenAPI)-절약을 위해 test용으로 한번만사용 - 코드엔 이어지게 구현

In [96]:
input_text

['본인의 금융상품 이해도를 평가할 수 있는 질문',
 ' 위기 상황에서 고객 응대를 어떻게 처리할 것인지 구체적인 사례를 들어 설명해달라는 질문',
 ' 팀 내 갈등 상황을 경험했을 때 어떻게 조정하고 협력했는지 알려달라는 질문']

### TTS(Text to Speech)
- 면접관의 질문 Text -> Speech로 변경

##### open AI test

In [97]:
# from openai import OpenAI

# client = OpenAI()

# # with가 한줄임
# for questinon_num, text in enumerate(input_text):
#     with client.audio.speech.with_streaming_response.create(
#         model = 'gpt-4o-mini-tts',
#         voice = 'nova', ### voice 정해줄 수 있음
#         input=input_text
#     ) as response:
#         response.stream_to_file('{questinon_num}_review_qustion.mp3')

##### gTTS Test

!pip install pydub

In [98]:
# !pip install imageio-ffmpeg

In [99]:
from gtts import gTTS
from pydub import AudioSegment
import imageio_ffmpeg as ffmpeg
import os

folder_path = f"C:/skn24/수업자료/08_large_language_model/01_open_ai/reviewer"
speed = 1.2 
ffmpeg_dir = r"C:\Users\Playdata\Downloads\ffmpeg-2026-03-15-git-6ba0b59d8b-essentials_build\ffmpeg-2026-03-15-git-6ba0b59d8b-essentials_build\bin"

# 환경변수에 강제 등록
# os.environ["PATH"] += os.pathsep + ffmpeg_dir

# 다시 지정
AudioSegment.converter = os.path.join(ffmpeg_dir, "ffmpeg.exe")
AudioSegment.ffprobe   = os.path.join(ffmpeg_dir, "ffprobe.exe")
for question_num, text in enumerate(input_text):
    tts_output = gTTS(text=text, lang='ko')
    tts_output.save(f'{folder_path}/{question_num+1}_review_qustion.mp3')


for i in range(1, len(input_text)+1):
    file_path = f"{folder_path}/{i}_review_qustion.mp3"
    
    try:
        sound = AudioSegment.from_file(file_path)

        faster_sound = sound._spawn(sound.raw_data, overrides={
            "frame_rate": int(sound.frame_rate * speed)
        }).set_frame_rate(sound.frame_rate)

        faster_sound.export(file_path, format="mp3")

        print(f"✅ 완료: {i}")

    except Exception as e:
        print(f"❌ 오류: {i} / {e}")

✅ 완료: 1
✅ 완료: 2
✅ 완료: 3


In [101]:
from IPython.display import Audio
import time 
print('질문 1')
display(Audio('./reviewer/1_review_qustion.mp3', autoplay=True))
display(Audio('./reviewer/2_review_qustion.mp3', autoplay=True))


질문 1


In [102]:
import speech_recognition as sr


recognizer = sr.Recognizer()
recognizer.energy_threshold = 300
recognizer.dynamic_energy_threshold = True

full_text = ""

while True:
    with sr.Microphone() as source:
        recognizer.adjust_for_ambient_noise(source, duration=1)
        print('마지막엔 이상입니다로 대답을 완료해주세요.')

        try:
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=20)

            text = recognizer.recognize_google(audio, language='ko-KR')
            print("인식 결과:", text)

            if '이상입니다' in text:
                text = text.replace('이상입니다', '')
                full_text += " " + text
                print("종료합니다.")
                break
            else:
                full_text += " " + text

        except sr.WaitTimeoutError:
            print("말이 없습니다.")
        except sr.UnknownValueError:
            print("음성을 이해하지 못했습니다.")
        except sr.RequestError as e:
            print(f"API 오류: {e}")

print("\n최종 저장 텍스트:")
print(full_text)

마지막엔 이상입니다로 대답을 완료해주세요.
인식 결과: 글자로 은행에서 근무하다 보면은 많은 수의
마지막엔 이상입니다로 대답을 완료해주세요.
인식 결과: 상황이 발생할 수 있습니다 그 과정에서
마지막엔 이상입니다로 대답을 완료해주세요.
인식 결과: 너무 어려운데 나 준비하니까
마지막엔 이상입니다로 대답을 완료해주세요.
인식 결과: 기사입니다
마지막엔 이상입니다로 대답을 완료해주세요.
인식 결과: 기사입니다 기사입니다
마지막엔 이상입니다로 대답을 완료해주세요.
인식 결과: 이상입니다
종료합니다.

최종 저장 텍스트:
 글자로 은행에서 근무하다 보면은 많은 수의 상황이 발생할 수 있습니다 그 과정에서 너무 어려운데 나 준비하니까 기사입니다 기사입니다 기사입니다 


In [110]:
full_text

' 글자로 은행에서 근무하다 보면은 많은 수의 상황이 발생할 수 있습니다 그 과정에서 너무 어려운데 나 준비하니까 기사입니다 기사입니다 기사입니다 '

##### 예시답변 
- AI 서비스 기획 시 데이터 윤리 문제를 어떻게 다루시나요?',
    - AI 서비스 기획 시 데이터 윤리는 개인정보 최소 수집, 익명화, 그리고 편향·차별 여부를 사전에 검증하는 프로세스를 포함해 관리합니다. 또한 데이터 활용 목적과 범위를 명확히 정의하고 사용자에게 투명하게 공개하는 것을 중요하게 생각합니다.

- Q2. 모델 성능 개선을 위해 주로 사용하는 방법론은 무엇인가요?'
    - 모델 성능 개선은 데이터 품질 개선과 하이퍼파라미터 튜닝을 기본으로, 에러 분석 기반 반복 개선을 주로 사용합니다. 필요 시 앙상블이나 파인튜닝을 통해 실제 서비스 환경에 맞게 성능을 최적화합니다.

In [ ]:
import json

query = full_text  # 교정할 대상 기사제목(원본)
# # 당신은 아주 깔끔하고 꼼꼼한 데이터분석을 수행하는 회사의 부사장이며,

system_instruction = """
# 당신은 아주 깔끔하고 꼼꼼한 데이터분석을 수행하는 회사의 부사장이며,
친절하고 구체적인 어드바이스를 제공하는 임원이다.

다음 기준으로 면접 피드백을 작성하라:
1. 논리 구조
2. 데이터 기반 사고
3. 커뮤니케이션 명확성
4. 개선 방향 (구체적)

반드시 아래 JSON 형식으로만 응답하라:
{
  "feedback": [
    "피드백1",
    "피드백2",
    "피드백3",
    "피드백4"
  ]
}

각 피드백은 짧고 명확하게 작성하라.
"""

# ✅ user_message 그대로 유지
user_message = f'{query} 내가 한 답변을 갖고 면접 피드백을 진행해줘'

feed_back = client.chat.completions.create(
    model='gpt-4.1-mini',
    messages=[
        {
            'role': 'system',
            'content': system_instruction
        },
        {
            'role': 'user',
            'content': user_message
        }
    ],
    response_format={"type": "json_object"},  # 🔥 핵심
    temperature=0.2,  # 🔥 안정성 위해 낮춤
    max_tokens=1024
)

# ✅ JSON 파싱
data = json.loads(feed_back.choices[0].message.content)

feed_back_text = data["feedback"]

print(feed_back_text)

['문장을 명확하고 간결하게 작성하세요.', '중복된 표현은 피하고 핵심만 전달하세요.', '면접 답변은 구체적인 경험과 사례를 포함하세요.', '답변 내용이 일관되고 논리적으로 연결되도록 하세요.']


##### 피드백 음성본

In [113]:

speed = 1.2  

# 다시 지정
AudioSegment.converter = os.path.join(ffmpeg_dir, "ffmpeg.exe")
AudioSegment.ffprobe   = os.path.join(ffmpeg_dir, "ffprobe.exe")
for feed_back_num, text in enumerate(feed_back_text):
    tts_output = gTTS(text=text, lang='ko')
    tts_output.save(f'{folder_path}/{feed_back_num+1}_feed_back.mp3')


for i in range(1, len(feed_back_text)+1):
    file_path = f"{folder_path}/{i}_feed_back.mp3"
    
    try:
        sound = AudioSegment.from_file(file_path)

        faster_sound = sound._spawn(sound.raw_data, overrides={
            "frame_rate": int(sound.frame_rate * speed)
        }).set_frame_rate(sound.frame_rate)

        faster_sound.export(file_path, format="mp3")

        print(f"✅ 완료: {i}")

    except Exception as e:
        print(f"❌ 오류: {i} / {e}")

✅ 완료: 1
✅ 완료: 2
✅ 완료: 3
✅ 완료: 4


In [115]:
display(Audio('./reviewer/1_feed_back.mp3', autoplay=True))
display(Audio('./reviewer/2_feed_back.mp3', autoplay=True))
display(Audio('./reviewer/3_feed_back.mp3', autoplay=True))
display(Audio('./reviewer/4_feed_back.mp3', autoplay=True))